# get similarity and coherence

In [22]:
import os
import re
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

In [23]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
pilot2_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = pilot2_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [24]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(pilot2_dir + model_path, binary=True)

## 데이터셋 준비

In [25]:
# 1_check_words파일에서 가공해서 저장한 데이터를 열어서 pilot_data에 테이블형태로 저장. 
# index_col=0: index를 표시해주는 컬럼을 사용하지 않겠다.
# keep_default_na=False: Nan값을 유지하지않겠다. ( 그냥 비어있는 문자열로 인식 )
pilot_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', index_col=0, keep_default_na=False)
pilot_data.fillna('') # 혹시 비어있는 칸이 있다면, 비어있는 문자열로 변경
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend21,friend22,friend23,friend24,friend25,friend26,friend27,friend28,friend29,friend30
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,burned down,scary,movies,knocked up,Tinsletown,christmas tree,holiday,decorate,fun,party
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time space,possibilites,infinite,time,...,transparent,glass,shattered,repair,meaningful,bond,glue,all together,matters,wonders
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,said,dressing,clothes,...,sunshine,warmth,blanket,heavy,body,muscle,protein,shake,dance,jump


In [26]:
seed_words = ['key', 'money', 'friend'] # 정해진 시드 단어들을 배열에 저장
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(pilot_data) # 58명의 피험자
n_dim_of_vector = 300 # 300차원

In [27]:
for seed_word in seed_words: ##### 차례로 seed_word에 key, money, friend가 저장됨
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # ex) ['key1', 'key2', ... , 'key30']

    for column in word_columns: ##### 차례로 column에 데이터의 컬럼 하나씩 저장됨. ex) key1
        pilot_data[column + '_vec'] = np.empty(n_subject, dtype=object) # 각 컬럼 벡터의 field 생성

        for i_subject in range(n_subject): ##### 차례로 피험자 숫자ID(0~57)
            try: 
                response_word = pilot_data.iloc[i_subject][column] # 피험자가 응답한 단어

                if isinstance(response_word, str): # isinstance(변수, type): 그 변수가 해당 type이면 true, 아니면 false.
                    response_words = response_word.split() # split(): ' '를 기준으로 단어를 분리해서 배열로 만듦. ex) jet lag -> ['jet', 'lag']
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]

                    vec_word2vec = np.zeros((n_dim_of_vector, 0)) # 응답 단어의 벡터를 저장할 빈 행렬 생성

                    for i_el in range(len(response_words)): # 위에서 split()해서 배열이 생성되므로 for 반복문을 돌리면서 각각을 word2vec모델에 넣어서 벡터값을 얻음
                        # try - except: 예외처리하는 문법. try내에 있는 코드가 실패하면, except에 있는 코드가 실행됨.
                        try:
                            vec_word2vec_in = word2vec_model[response_words[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_words[i_el].capitalize()]    
                        
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1)) # reshape((n, m)): 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in)) # np.hstack(): 지정된 두개의 벡터값들을 수직으로 붙임. 

                    pilot_data[column + '_vec'][i_subject] = np.array(vec_word2vec)  # numpy배열로 변환해서, 해당 위치에 벡터값 저장.
            except:
                pass

/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_3537/278605298.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_data[column + '_vec'][i_subject] = np.array(vec_word2vec)  # numpy배열로 변환해서, 해당 위치에 벡터값 저장.
/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_3537/278605298.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pilot_data[column + '_vec'][i_subject] = np.array(vec_word2vec)  # numpy배열로 변환해서, 해당 위치에 벡터값 저장.
/var/folders/rg/snggr9dd58q1xqcclvb4k8yr0000gn/T/ipykernel_3537/278605298.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See t

In [28]:
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,friend21_vec,friend22_vec,friend23_vec,friend24_vec,friend25_vec,friend26_vec,friend27_vec,friend28_vec,friend29_vec,friend30_vec
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,"[[0.306640625, 0.0245361328125], [-0.005767822...","[[0.171875], [-0.1240234375], [0.1748046875], ...","[[0.01361083984375], [0.138671875], [-0.163085...","[[0.10009765625, 0.1201171875], [-0.0094604492...","[[0.1328125], [0.055908203125], [-0.1982421875...","[[-0.1689453125, 0.484375], [0.03466796875, 0....","[[0.275390625], [0.1201171875], [-0.212890625]...","[[0.08642578125], [0.047607421875], [-0.117675...","[[0.0791015625], [-0.1201171875], [-0.09423828...","[[-0.09130859375], [-0.0869140625], [-0.012084..."
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time space,possibilites,infinite,time,...,"[[-0.287109375], [0.0179443359375], [-0.205078...","[[-0.2236328125], [0.1240234375], [-0.09130859...","[[0.033203125], [0.1953125], [0.0260009765625]...","[[-0.03173828125], [0.392578125], [-0.10302734...","[[-0.1279296875], [-0.035400390625], [-0.08007...","[[0.1923828125], [-0.09619140625], [0.18164062...","[[0.056640625], [0.047607421875], [-0.02124023...","[[-0.0078125, -0.1083984375], [-0.027954101562...","[[0.0308837890625], [0.25390625], [-0.04711914...","[[0.1708984375], [0.11376953125], [0.050292968..."
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,said,dressing,clothes,...,"[[0.00823974609375], [0.07373046875], [0.04321...","[[0.2490234375], [0.13671875], [0.025390625], ...","[[0.02783203125], [-0.062255859375], [0.222656...","[[0.2890625], [0.306640625], [-0.15234375], [0...","[[-0.01129150390625], [-0.020751953125], [0.38...","[[0.46875], [0.279296875], [-0.006988525390625...","[[-0.12109375], [0.00136566162109375], [-0.007...","[[-0.0830078125], [-0.11474609375], [-0.056640...","[[0.18359375], [-0.318359375], [0.205078125], ...","[[0.052001953125], [0.0361328125], [-0.1035156..."


# similarity - coherence

- 피험자가 응답한 모든 단어 ↔ `key` 간의 similarity 
- 피험자가 응답한 모든 단어 ↔ `money` 간의 similarity
- 피험자가 응답한 모든 단어 ↔ `friend` 간의 similarity \
\
\
-> coherence값 구함

In [29]:
target_words = ['key', 'money', 'friend'] # seed_words와 동일

# similarity 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)]
        for column in word_columns: 
                column_name = f'similarity_{column}_{target_word}'
                pilot_data = pilot_data.assign(**{column_name: None})
# coherence 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        column_name = f'coherence_{seed_word}_{target_word}'
        pilot_data = pilot_data.assign(**{column_name: None})

pilot_data.columns

Index(['Prolific_ID', 'subject', 'key1', 'key2', 'key3', 'key4', 'key5',
       'key6', 'key7', 'key8',
       ...
       'similarity_friend30_friend', 'coherence_key_key',
       'coherence_money_key', 'coherence_friend_key', 'coherence_key_money',
       'coherence_money_money', 'coherence_friend_money',
       'coherence_key_friend', 'coherence_money_friend',
       'coherence_friend_friend'],
      dtype='object', length=461)

In [30]:
for seed_word in seed_words: ##### 차례로 seed_word에 key, money, friend가 저장됨
    # coherence값 저장할 배열 초기화
    coherences_key_per_sub = [] 
    coherences_money_per_sub = []
    coherences_friend_per_sub = []

    word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)] # ex) ['key1', 'key2', ... , 'key30']

    for i_subject in range(n_subject): ##### 차례로 피험자 숫자ID(0~57)
        # similarity값 저장할 배열 초기화
        each_seed_similarities_key = []  
        each_seed_similarities_money = []  
        each_seed_similarities_friend = []

        for column in word_columns: ##### 차례로 column에 데이터의 컬럼 하나씩 저장됨. ex) key1
            try:
                # 각 타겟 단어와 응답 단어 90개의 유사도 추출
                similarity_between_key = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'key')
                similarity_between_money = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'money')
                similarity_between_friend = word2vec_model.similarity(pilot_data.iloc[i_subject][column], 'friend')
                # 추출한 유사도를 해당 테이블 위치에 저장
                pilot_data.at[i_subject, f'similarity_{column}_key'] = similarity_between_key
                pilot_data.at[i_subject, f'similarity_{column}_money'] = similarity_between_money
                pilot_data.at[i_subject, f'similarity_{column}_friend'] = similarity_between_friend
                # 배열에 넣기
                each_seed_similarities_key.append(similarity_between_key) # 30개
                each_seed_similarities_money.append(similarity_between_money) # 30개
                each_seed_similarities_friend.append(similarity_between_friend) # 30개
            except:
                continue

        # 해당 피험자가 하나의 seed에 답한 30개의 응답단어의 유사도의 coherence값
        # i. 유사도의 평균값 (O)
        key_coherence_of_seed_per_sub = np.mean(each_seed_similarities_key) 
        money_coherence_of_seed_per_sub = np.mean(each_seed_similarities_money) 
        friend_coherence_of_seed_per_sub = np.mean(each_seed_similarities_friend)
        # ii. gensim의 CoherenceModel을 고려했으나, 토픽 최적화에 주로 사용하는 듯함 (X)

        # 해당 피험자에 대한, 그 seed단어 각각(3개)에 대한 coherence값
        # 피험자마자 9개 값( key,money,friend - key,money,friend 조합)
        coherences_key_per_sub.append(key_coherence_of_seed_per_sub)
        coherences_money_per_sub.append(money_coherence_of_seed_per_sub)
        coherences_friend_per_sub.append(friend_coherence_of_seed_per_sub)

        # 계산한 coherence를 해당 테이블 위치에 저장
        pilot_data.at[i_subject, f'coherence_{seed_word}_key'] = key_coherence_of_seed_per_sub
        pilot_data.at[i_subject, f'coherence_{seed_word}_money'] = money_coherence_of_seed_per_sub
        pilot_data.at[i_subject, f'coherence_{seed_word}_friend'] = friend_coherence_of_seed_per_sub



In [31]:
pilot_data[0:3]

,Prolific_ID,subject,key1,key2,key3,key4,key5,key6,key7,key8,...,similarity_friend30_friend,coherence_key_key,coherence_money_key,coherence_friend_key,coherence_key_money,coherence_money_money,coherence_friend_money,coherence_key_friend,coherence_money_friend,coherence_friend_friend
0,5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,...,0.199907,0.041143,0.03042,0.037806,0.102066,0.143791,0.09695,0.075201,0.063515,0.146998
1,5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time space,possibilites,infinite,time,...,0.171929,0.096018,0.053525,0.081238,0.140075,0.126333,0.10836,0.120077,0.125166,0.105842
2,5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,said,dressing,clothes,...,0.058081,0.079032,0.057519,0.049724,0.117155,0.099424,0.145488,0.169555,0.132149,0.082121


In [32]:
# vector 컬럼들 드롭 ( csv용량이 너무 커지는 것을 방지 )
drop_columns = pilot_data.columns[92:182] 
pilot_data = pilot_data.drop(drop_columns, axis='columns')

# 단어 있는 버전 csv 저장
pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data_with_words.csv', index=None)

In [33]:
# 단어 컬럼들 드롭
drop_columns = pilot_data.columns[2:92] 
pilot_data = pilot_data.drop(drop_columns, axis='columns')

# 단어 없이 coherence만 있는 버전 csv 저장
pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data.csv', index=None)